In [47]:
import numpy as np
import pandas as pd
import csv
import random
from sklearn.linear_model import LogisticRegression
import sys
import os

sys.path.insert(0, "../../../Modules")

pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 10)
random.seed(0)
np.random.seed(0)

from utils import *

enc = 'utf-8'

In [48]:
import warnings
warnings.filterwarnings("ignore")

In [49]:
NUTS0 = 'GR'
NUTS2 = 'CMacedonia'

In [50]:
YEAR = 2023

In [51]:
base_path = f'../../data/{NUTS2}/results/'

In [52]:
file_list = os.listdir(base_path)
file_list.sort()
file_list

['GR_CMacedonia_Results_2010-2022.csv',
 'GR_CMacedonia_Results_2023-July-1st.csv',
 'GR_CMacedonia_Results_2023-July-2nd.csv',
 'GR_CMacedonia_Results_2023-June-1st.csv',
 'GR_CMacedonia_Results_2023-June-2nd.csv',
 'GR_CMacedonia_Results_2023-May-1st.csv',
 'GR_CMacedonia_Results_2023-May-2nd.csv']

In [53]:
result_files = []

for file in file_list:
    data = pd.read_csv(f'{base_path}/{file}', encoding='utf-8')
    data.drop(columns=['x', 'y'], inplace=True)
    data.rename(columns={"probability": "score", "municipality": "lau1"}, inplace=True)
    result_files.append(data)

In [54]:
result_files[0].drop(columns = ['case'], inplace = True)

In [57]:
result_files[1].drop(columns = ['risk_class'], inplace = True)

In [60]:
result_files[2].drop(columns = ['risk_class'], inplace = True)

In [62]:
result_files[3].drop(columns = ['Class Number - (Relative Risk Level)', 'Class Number - (Risk Level)'], inplace = True)

In [64]:
result_files[4].drop(columns = ['Class Number - (Relative Risk Level)', 'Class Number - (Risk Level)'], inplace = True)

In [66]:
result_files[5].drop(columns = ['Class Number - (Relative Risk Level)', 'Class Number - (Risk Level)'], inplace = True)

In [68]:
result_files[6].drop(columns = ['Class Number - (Relative Risk Level)', 'Class Number - (Risk Level)'], inplace = True)

In [69]:
concated = pd.concat(result_files, ignore_index=True, axis = 0)

In [71]:
concated.month.unique()

array([ 8,  9,  7, 10,  6, 11,  5], dtype=int64)

In [72]:
zero_score = {'lau1': 'αγνωστη', 'day': 0, 'month': 0, 'year': 0, 'score': 0}
one_score = {'lau1': 'αγνωστη', 'day': 0, 'month': 0, 'year': 0, 'score': 1}

In [73]:
concated

,lau1,day,month,year,score
0,θεσσαλονικης,16,8,2021,0.989443
1,αλεξανδρειας,16,8,2021,0.988773
2,αλεξανδρειας,1,8,2021,0.988579
3,δελτα,1,9,2012,0.986375
4,αλεξανδρειας,1,9,2021,0.986129
...,...,...,...,...,...
6151,πυδνας κολινδρου,16,5,2023,0.001364
6152,νεαπολης συκεων,16,5,2023,0.001020
6153,πολυγυρου,16,5,2023,0.000984
6154,αριστοτελη,16,5,2023,0.000667


In [74]:
concated_og = concated.copy()

In [75]:
# high_values_rows = concated.loc[concated['month'].isin([7, 8])]
# concated = pd.concat([concated, high_values_rows], ignore_index=True)

In [76]:
# high_score_rows = concated.loc[concated['score'] >= 0.8]
# very_high_score_rows = concated.loc[concated['score'] >= 0.9]

# concated = pd.concat([concated, high_score_rows], ignore_index=True)
# concated = pd.concat([concated, high_score_rows], ignore_index=True)
# concated = pd.concat([concated, very_high_score_rows], ignore_index=True)

In [77]:
concated = concated.append(zero_score, ignore_index = True)
concated = concated.append(one_score, ignore_index = True)

In [78]:
concated

,lau1,day,month,year,score
0,θεσσαλονικης,16,8,2021,0.989443
1,αλεξανδρειας,16,8,2021,0.988773
2,αλεξανδρειας,1,8,2021,0.988579
3,δελτα,1,9,2012,0.986375
4,αλεξανδρειας,1,9,2021,0.986129
...,...,...,...,...,...
6153,πολυγυρου,16,5,2023,0.000984
6154,αριστοτελη,16,5,2023,0.000667
6155,σιθωνιας,16,5,2023,0.000658
6156,αγνωστη,0,0,0,0.000000


In [79]:
out, bins = pd.qcut(concated['score'], 10, retbins= True, labels=range(10))

In [80]:
bins

array([0.00000000e+00, 7.31429516e-04, 3.10847833e-03, 1.13224327e-02,
       3.75778140e-02, 8.95396352e-02, 1.91261314e-01, 3.53651982e-01,
       5.66315218e-01, 7.84838005e-01, 1.00000000e+00])

In [27]:
bins

array([0.        , 0.02038365, 0.22274505, 0.5478774 , 0.80295725,
       0.84222879, 0.8820724 , 0.91557888, 0.96      , 0.98      ,
       1.        ])

In [82]:
concated_og['risk_class'] = pd.cut(concated_og['score'], bins=bins, labels=False, include_lowest=True)

In [83]:
concated_og.risk_class.value_counts()

8    616
7    616
5    616
4    616
2    616
1    616
9    615
6    615
3    615
0    615
Name: risk_class, dtype: int64

In [81]:
bins_formatted = [ '%.4f' % elem for elem in bins]
print(bins_formatted)

['0.0000', '0.0007', '0.0031', '0.0113', '0.0376', '0.0895', '0.1913', '0.3537', '0.5663', '0.7848', '1.0000']


In [84]:
scores = concated['score'].tolist()

In [85]:
bins = bins.tolist()

In [86]:
scores_path = f'../../data/{NUTS2}/{NUTS0}_{NUTS2}_Scores_{YEAR}_(7).csv'

with open(scores_path, mode='w', newline='') as file:
    writer = csv.writer(file)
    writer.writerow(scores)

In [87]:
bins_path = f'../../data/{NUTS2}/{NUTS0}_{NUTS2}_Bins_{YEAR}_(7).csv'

with open(bins_path, mode='w', newline='') as file:
    writer = csv.writer(file)
    writer.writerow(bins)